In [4]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main/01-intro/minsearch.py

--2025-02-24 16:53:31--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main/01-intro/minsearch.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3832 (3.7K) [text/plain]
Saving to: 'minsearch.py.1'

     0K ...                                                   100%  650K=0.006s

2025-02-24 16:53:32 (650 KB/s) - 'minsearch.py.1' saved [3832/3832]



In [5]:
import minsearch

In [6]:
import json 

In [7]:
with open('documents.json', 'rt') as f_in:
    docs_raw = json.load(f_in)


In [8]:
documents = []

for course_dict in docs_raw:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        documents.append(doc)

In [9]:
documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [10]:
index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

In [11]:
q = 'the course has already started, can I still enroll?'

In [12]:
index.fit(documents)

In [13]:
boost = {'question':3.0,'section':0.5}

results = index.search(
    query = q,
    boost_dict = boost,
    num_results=5
)

results

[{'text': 'Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course.\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers’ Projects by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'section': 'General course-related questions',
  'question': 'The course has already started. Can I still join it?',
  'course': 'machine-learning-zoomcamp'},
 {'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Yes, we will keep all the materials after the cour

In [40]:
import os
from getpass import getpass

if "MISTRAL_API_KEY" not in os.environ:
    os.environ["MISTRAL_API_KEY"] = getpass("Enter your API KEY")


In [42]:
from langchain_mistralai import ChatMistralAI

# Instantiate the Mistral model
llm = ChatMistralAI(
    model="mistral-large-latest",
    temperature=0,
    max_retries=2,
    # other parameters...
)

# Prepare the messages
messages = [
    {"role": "user", "content": q},
]

# Get the response from the Mistral model
response = llm.invoke(messages)

# Extract the content of the response
response_content = response.content
print(response_content)


Whether you can still enroll in a course that has already started depends on several factors, including the policies of the institution or platform offering the course, the nature of the course, and the availability of spots. Here are some steps you can take to find out:

1. **Check the Course Website or Platform**: Look for information on the course's website or the platform where it is hosted. There may be details about late enrollment policies.

2. **Contact the Instructor or Administrator**: Reach out to the course instructor or the administrative office of the institution. They can provide specific information about whether late enrollment is allowed and what the process entails.

3. **Review the Syllabus**: If the syllabus is available, it might include information about late enrollment and any deadlines or requirements.

4. **Consider the Impact**: Think about how starting late might affect your ability to catch up. Some courses may have strict attendance policies or require par

In [44]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5
    )

    return results

In [46]:
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT: 
{context}
""".strip()

    context = ""
    
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [48]:
from langchain_mistralai import ChatMistralAI

# Define the function to invoke the Mistral model
def llm(prompt):
    llm_instance = ChatMistralAI(
        model="mistral-large-latest",
        temperature=0,
        max_retries=2,
        # other parameters...
    )
    response = llm_instance.invoke([{"role": "user", "content": prompt}])
    return response.content

In [50]:
query = 'how do I run kafka?'

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [52]:
rag(query)

'To run Kafka, you can follow these instructions based on the context provided:\n\nFor Java Kafka:\n1. Navigate to the project directory.\n2. Run the following command in the terminal:\n   ```\n   java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n   ```\n\nFor Python Kafka:\n1. Create a virtual environment and install the necessary packages.\n   ```\n   python -m venv env\n   source env/bin/activate  # For Windows, use env\\Scripts\\activate\n   pip install -r ../requirements.txt\n   ```\n2. Ensure that the Docker images are up and running before executing the Python files.\n\nIf you encounter the "ModuleNotFoundError: No module named \'kafka.vendor.six.moves\'" error, it is suggested to use the `kafka-python-ng` package instead:\n```\npip install kafka-python-ng\n```\n\nFor permission issues with `build.sh`, run the following command in the terminal in the same directory (/docker/spark):\n```\nchmod +x build.sh\n```'

In [53]:
rag('the course has already started, can I still enroll?')

"Yes, you can still enroll in the course even after it has started. Even if you don't register, you're still eligible to submit the homeworks. However, be aware that there will be deadlines for turning in the final projects, so it's important not to leave everything for the last minute."

In [2]:
from elasticsearch import Elasticsearch

In [56]:
es_client = Elasticsearch('http://localhost:9200')

In [ ]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"} 
        }
    }
}
index_name = "course-questions"
es_client.indices.create(index = index_name,body = index_settings)

In [ ]:
search_query = {
    "size": 5,
    "query": {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query,
                    "fields": ["question^3", "text", "section"],
                    "type": "best_fields"
                }
            },
            "filter": {
                "term": {
                    "course": "data-engineering-zoomcamp"
                }
            }
        }
    }
}